In [0]:
loans_def_raw_df = spark.read \
.format("csv") \
.option("header", True) \
.option("inferSchema",True) \
.load("/Volumes/finance_domain_project/default/raw_data/Raw/loans_defaulters_csv")


In [0]:
loans_def_raw_df.createOrReplaceTempView("loan_defaulters")

In [0]:
display(spark.sql("select distinct(delinq_2yrs) from loan_defaulters"))

In [0]:
loan_defaulters_schema = "member_id string, delinq_2yrs float, delinq_amnt float, pub_rec float, pub_rec_bankruptcies float,inq_last_6mths float,total_rec_late_fee float,mths_since_last_delinq float,mths_since_last_record float "

In [0]:
loans_def_raw_df = spark.read \
.format("csv") \
.option("header", True) \
.schema(loan_defaulters_schema) \
.load("/Volumes/finance_domain_project/default/raw_data/Raw/loans_defaulters_csv")



In [0]:
loans_def_raw_df.createOrReplaceTempView("loan_defaulters")

In [0]:
spark.sql("select delinq_2yrs, count(*) as total from loan_defaulters group by delinq_2yrs order by total desc").show(40)

In [0]:
from pyspark.sql.functions import col

In [0]:
loans_def_processed_def = loans_def_raw_df.withColumn("delinq_2yrs",col("delinq_2yrs").cast("integer")).fillna(0,subset=["delinq_2yrs"])

In [0]:
loans_def_processed_def.createOrReplaceTempView("loan_defaulters")

In [0]:
display(spark.sql("select count(*) from loan_defaulters where delinq_2yrs is null"))

In [0]:
spark.sql("select delinq_2yrs, count(*) as total from loan_defaulters group by delinq_2yrs order by total desc").show(40)

In [0]:
loans_def_delinq_df = spark.sql("select member_id, delinq_2yrs,delinq_amnt, int(mths_since_last_delinq) from loan_defaulters where delinq_2yrs>0 or mths_since_last_delinq > 0")

In [0]:
loans_def_records_enq_df=spark.sql("select member_id from loan_defaulters where pub_rec > 0.0 or pub_rec_bankruptcies > 0.0 or inq_last_6mths > 0.0")

In [0]:
loans_def_delinq_df.write \
.option("header",True) \
.format("csv") \
.mode("overwrite") \
.option("path","/Volumes/finance_domain_project/default/raw_data/Cleaned/loans_defaulters_delinq_csv") \
.save()

In [0]:
loans_def_delinq_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path","/Volumes/finance_domain_project/default/raw_data/Cleaned/loans_defaulters_delinq_parquet") \
.save()


In [0]:
loans_def_records_enq_df.write \
.option("header",True) \
.format("csv") \
.mode("overwrite") \
.option("path","/Volumes/finance_domain_project/default/raw_data/Cleaned/loans_defaulters_records_enq_csv") \
.save()


In [0]:
loans_def_records_enq_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path","/Volumes/finance_domain_project/default/raw_data/Cleaned/loans_defaulters_records_enq_parquet") \
.save()


In [0]:
loans_def_processed_df = loans_def_raw_df.withColumn("delinq_2yrs", col("delinq_2yrs").cast("integer")).fillna(0,subset=["delinq_2yrs"])

In [0]:
loans_def_processed_pub_rec_df=loans_def_processed_df.withColumn("pub_rec", col("pub_rec").cast("integer")).fillna(0,subset=["pub_rec"])

In [0]:
loans_def_processed_pub_rec_bankruptcies_df=loans_def_processed_pub_rec_df.withColumn("pub_rec_bankruptcies", col("pub_rec_bankruptcies").cast("integer")).fillna(0,subset=["pub_rec_bankruptcies"])

In [0]:
loans_def_processed_inq_last_6mths_df=loans_def_processed_pub_rec_bankruptcies_df.withColumn("inq_last_6mths", col("inq_last_6mths").cast("integer")).fillna(0,subset=["inq_last_6mths"])

In [0]:
loans_def_processed_inq_last_6mths_df.createOrReplaceTempView("loan_defaulters")

In [0]:
loans_def_detail_records_enq_df= spark.sql("select member_id, pub_rec, pub_rec_bankruptcies,inq_last_6mths from loan_defaulters")

In [0]:
loans_def_detail_records_enq_df.write \
.option("header",True) \
.format("csv") \
.mode("overwrite") \
.option("path","/Volumes/finance_domain_project/default/raw_data/Cleaned/loans_defaulters_detail_records_enq_csv") \
.save()


In [0]:
loans_def_detail_records_enq_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path","/Volumes/finance_domain_project/default/raw_data/Cleaned/loans_defaulters_detail_records_enq_parquet") \
.save()
